In [1]:
#!pip install optuna

In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.metrics import mean_squared_error as MSE
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split,KFold
import optuna
from sklearn.preprocessing import StandardScaler,TargetEncoder, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Dropout,  BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.simplefilter('ignore')

C:\Users\dipty\AppData\Local\Temp\ipykernel_10752\2311569042.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
c:\Users\dipty\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df=pd.read_csv('train.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3994318 entries, 0 to 3994317
Data columns (total 11 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   Brand                 object 
 2   Material              object 
 3   Size                  object 
 4   Compartments          float64
 5   Laptop Compartment    object 
 6   Waterproof            object 
 7   Style                 object 
 8   Color                 object 
 9   Weight Capacity (kg)  float64
 10  Price                 float64
dtypes: float64(3), int64(1), object(7)
memory usage: 335.2+ MB


In [5]:
cat_cols=['Brand','Material','Size', 'Laptop Compartment', 'Waterproof', 'Style', 'Color']
num_cols=['Compartments', 'Weight Capacity (kg)']

In [6]:
df[cat_cols] = df[cat_cols].fillna('Unknown')
target_encoder = TargetEncoder()
df[cat_cols] = target_encoder.fit_transform(df[cat_cols], df['Price'])
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
scaler1 = StandardScaler()
scaler2 = StandardScaler()
y=df.Price
X = df[["Weight Capacity (kg)"]].values  
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)
X_poly = scaler1.fit_transform(X_poly)
df=df.drop(columns=['id','Price'])
X = scaler2.fit_transform(df)

In [7]:
X = np.hstack([X, X_poly])

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=95)

from tensorflow.keras.losses import MeanSquaredError

class RMSELoss(tf.keras.losses.Loss):
    def __init__(self):
        super().__init__()

    def call(self, y_true, y_pred):
        return tf.sqrt(MeanSquaredError()(y_true, y_pred))  


def objective(trial):
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    num_layers = trial.suggest_int("num_layers", 2, 4)
    neurons = trial.suggest_int("neurons", 100,400)
    dropout_rate = trial.suggest_uniform("dropout_rate", 0.2, 0.5)
    l1_l2_reg = trial.suggest_loguniform("l1_l2_reg", 1e-5, 1e-3)
    model = Sequential()
    model.add(Dense(neurons, input_shape=(X_train.shape[1],), activation=LeakyReLU(), kernel_regularizer=l1_l2(l1=l1_l2_reg, l2=l1_l2_reg)))
    for _ in range(num_layers):
        model.add(Dense(neurons, activation=LeakyReLU(), kernel_regularizer=l1_l2(l1=l1_l2_reg, l2=l1_l2_reg)))
        model.add(Dropout(dropout_rate))  
    model.add(Dense(1))  
    model.compile(optimizer=Adam(learning_rate=lr), loss=RMSELoss())
    early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        batch_size=batch_size,
        epochs=25,
        callbacks=[early_stopping],
        verbose=1
    )
    val_loss = min(history.history["val_loss"])
    print({"lr":lr,"batch_size":batch_size,"num_layers":num_layers,"neurons":neurons,"dropout_rate":dropout_rate,"l1_l2_reg":l1_l2_reg})
    print(f"Trial {trial.number} - RMSE: {np.sqrt(val_loss)}")
    return np.sqrt(val_loss) 

In [ ]:
study = optuna.create_study(direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=70, show_progress_bar=True)

best_params = study.best_params
print("Best Hyperparameters:", best_params)

[I 2025-02-12 21:55:12,410] A new study created in memory with name: no-name-280568c0-f508-435f-a230-a5797f2e42d9
  0%|          | 0/70 [00:00<?, ?it/s]

Epoch 1/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 734s 12ms/step - loss: 40.5136 - val_loss: 39.3568
Epoch 2/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 1079s 17ms/step - loss: 39.5014 - val_loss: 39.1249
Epoch 3/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 956s 15ms/step - loss: 39.3673 - val_loss: 39.0723
Epoch 4/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 885s 13ms/step - loss: 39.2046 - val_loss: 39.0348
Epoch 5/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 787s 13ms/step - loss: 39.1414 - val_loss: 38.9059
Epoch 6/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 846s 13ms/step - loss: 39.0325 - val_loss: 38.8896
Epoch 7/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 915s 14ms/step - loss: 38.9964 - val_loss: 38.8510
Epoch 8/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 888s 14ms/step - loss: 38.9647 - val_loss: 38.8588
Epoch 9/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 873s 14ms/step - loss: 38.9420 - val_loss: 38.8465
Epoch 10/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 553s 8ms/step - loss: 38.9544 - val_loss: 38.8368
Epoch 11/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 302s

Best trial: 0. Best value: 6.22836:   1%|▏         | 1/70 [3:34:49<247:02:31, 12889.15s/it]

{'lr': 6.99614038754224e-05, 'batch_size': 32, 'num_layers': 3, 'neurons': 400, 'dropout_rate': 0.2850624193982175, 'l1_l2_reg': 2.3602508689229078e-05}
Trial 0 - RMSE: 6.228358221592156
Epoch 1/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 52s 3ms/step - loss: 40.5728 - val_loss: 39.2181
Epoch 2/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - loss: 39.6043 - val_loss: 39.4479
Epoch 3/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 49s 3ms/step - loss: 39.2709 - val_loss: 39.0879
Epoch 4/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 51s 3ms/step - loss: 39.3970 - val_loss: 38.9475
Epoch 5/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - loss: 39.3848 - val_loss: 38.9273
Epoch 6/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - loss: 39.8624 - val_loss: 38.9297
Epoch 7/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - loss: 39.4185 - val_loss: 40.2199
Epoch 8/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - loss: 39.2766 - val_loss: 42.5749
Epoch 9/25
15603/15603 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - loss: 39.5

Best trial: 0. Best value: 6.22836:   3%|▎         | 2/70 [3:47:24<108:38:19, 5751.46s/it] 

...

{'lr': 0.000833538902866484, 'batch_size': 64, 'num_layers': 4, 'neurons': 345, 'dropout_rate': 0.23851240669122348, 'l1_l2_reg': 8.986768509576342e-05}
Trial 18 - RMSE: 6.2355252306717315
Epoch 1/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 177s 3ms/step - loss: 43.8289 - val_loss: 39.1888
Epoch 2/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 175s 3ms/step - loss: 40.0427 - val_loss: 39.1078
Epoch 3/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 175s 3ms/step - loss: 39.8866 - val_loss: 39.1016
Epoch 4/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 175s 3ms/step - loss: 39.8282 - val_loss: 39.1040
Epoch 5/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 177s 3ms/step - loss: 39.7309 - val_loss: 39.0991
Epoch 6/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 179s 3ms/step - loss: 39.6579 - val_loss: 39.1087
Epoch 7/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 179s 3ms/step - loss: 39.5857 - val_loss: 39.0707
Epoch 8/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 184s 3ms/step - loss: 39.5826 - val_loss: 39.0696
Epoch 9/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 204s 3ms/step -

Best trial: 7. Best value: 6.22822:  29%|██▊       | 20/70 [19:17:13<52:46:29, 3799.80s/it]

{'lr': 1.1355786778875944e-05, 'batch_size': 32, 'num_layers': 4, 'neurons': 277, 'dropout_rate': 0.42000660136828993, 'l1_l2_reg': 1.7192992136906323e-05}
Trial 19 - RMSE: 6.243106244477502
Epoch 1/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 125s 2ms/step - loss: 43.6643 - val_loss: 39.2939
Epoch 2/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 122s 2ms/step - loss: 39.4581 - val_loss: 38.8968
Epoch 3/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 124s 2ms/step - loss: 39.1026 - val_loss: 38.8492
Epoch 4/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 125s 2ms/step - loss: 39.0565 - val_loss: 38.8270
Epoch 5/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 124s 2ms/step - loss: 39.0281 - val_loss: 38.8271
Epoch 6/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 125s 2ms/step - loss: 39.0350 - val_loss: 38.8256
Epoch 7/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 127s 2ms/step - loss: 39.0116 - val_loss: 38.8247
Epoch 8/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 127s 2ms/step - loss: 39.0055 - val_loss: 38.8252
Epoch 9/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 126s 2ms/step

Best trial: 7. Best value: 6.22822:  30%|███       | 21/70 [20:03:13<47:28:06, 3487.48s/it]

{'lr': 0.00014316314450934915, 'batch_size': 32, 'num_layers': 3, 'neurons': 206, 'dropout_rate': 0.34725023395104143, 'l1_l2_reg': 0.0009793099926393742}
Trial 20 - RMSE: 6.230923033986919
Epoch 1/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 197s 3ms/step - loss: 40.4505 - val_loss: 39.1297
Epoch 2/25
62412/62412 ━━━━━━━━━━━━━━━━━━━━ 173s 3ms/step - loss: 39.4826 - val_loss: 39.0418
Epoch 3/25
62408/62412 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39.2799

In [ ]:
[0,7,11,14,15,16]

In [ ]:
#best_params={'lr': 6.609214561627009e-05, 'batch_size': 32, 'num_layers': 2, 'neurons': 155, 'dropout_rate': 0.40275898803098964, 'l1_l2_reg': 2.5400182591058813e-05}
best_model = Sequential()
best_model.add(Dense(best_params["neurons"], input_shape=(X_train.shape[1],), activation=LeakyReLU(), kernel_regularizer=l1_l2(l1=best_params["l1_l2_reg"], l2=best_params["l1_l2_reg"])))

for _ in range(best_params["num_layers"]):
    best_model.add(Dense(best_params["neurons"], activation=LeakyReLU(), kernel_regularizer=l1_l2(l1=best_params["l1_l2_reg"], l2=best_params["l1_l2_reg"])))
    best_model.add(Dropout(best_params["dropout_rate"]))

best_model.add(Dense(1))
best_model.compile(optimizer=Adam(learning_rate=lr), loss=RMSELoss())
best_model.fit(X_train, y_train, epochs=25, batch_size=best_params["batch_size"], verbose=1, callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)])

test_loss = best_model.evaluate(X_test, y_test)
print(f"Final RMSE on Test Set: {test_loss}")

In [ ]:
df=pd.read_csv('test.csv')
df[cat_cols] = df[cat_cols].fillna('Unknown')
df[cat_cols] = target_encoder.transform(df[cat_cols])
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
X = df[["Weight Capacity (kg)"]].values  
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)
X_poly = scaler1.transform(X_poly)
df=df.drop(columns=['id'])
X = scaler2.transform(df)
X = np.hstack([X, X_poly])

In [ ]:
submission=pd.read_csv('sample_submission.csv')
submission['Price']=best_model.predict(X)
submission.set_index('id').to_csv('submissionDl.csv')

In [ ]:
df = study.trials_dataframe()
df.head()